PROGETTO 1: COSTRUIRE E ADDESTRARE UN MODELLO PER RICONOSCIMENTO CIFRE SCRITTE A MANO
MNIST Pro

- Ottimizzazine estrema dell'architettura: stacking dei layer convoluzionale e design dei kernel
- Tecniche di regolarizzazione avanzata: Batch Normalization, Dropout dinamico e Weight Decay. Più il modello è potente più rischia di memorizzare i dati di addestramento
- Raggiungimento del top performance: Learning Rate Scheduling e monitoraggio della convergenza.

Perchè la vecchia LeNet non basta più?

Ottimizzazione Estrema dell'Architettura
Oltre la struttura LeNet standard.
Per superare la soglia del 99% di accuratezza su MNIST, non è sufficiente una rete convoluzionale generica. Dobbiamo progettare una gerarchia di filtri che catturi dettagli microscopici e pattern globali, dobbiamo progettare una gerarchia di filtri che non veda solo forme, ma comprenda la microstruttura delle cifre.
L'idea è smette di pensare a layer singoli ma iniziare a pensare a blocchi funzionali. Metteremo più layer convoluzionali prima del pooling, questo aumenta la non  linearità
L'ottimizzazione dell'architettura prevede l'uso di blocchi 'Conv-Conv-Pool' invece del classico 'Conv-Pool', aumentando la profondità non lineare senza distruggere la risoluzione spaziale troppo velocemente.

Ma come si progetta un blocco così efficiente?

Design dei Filtri e Stacking
Strategia di profondità
- Stacking Convoluzionale: l'uso di due layer 'Conv2D' consecutivi prima del pooling permette di avere un campo ricettivo pià ampio con meno parametri rispetto a un singolo kernel grande.
- Feature Map Expansion: incremento progressivo del numero di filtri (es. 32, 64, 128) per codificare informazioni semantiche sempre pià astratte.
- Kernel 3x3: l'utilizzo esclusivo di kernel piccoli riduce il numero di pesi e facilita l'apprendimento tramite la sovrapposizione
- Global Average Pooling: sostituzione dei layer 'Flatten' troppo pesanti con il pooling globale per ridurre drasticamente il rischio di overfitting nel classificatore finale.

Attenzione perchè in questo processo la Risoluzione è il nostro processo più caro

Invarianza e Risoluzione
L'utilizzo del padding garantisce che l'informazione dei bordi dell'immagine non vada persa durante i numerosi passaggi convoluzionali. Utilizziamo il Padding Same, manteniamo la dimensione dell'immagine, affinchè i layer profondi abbiano ancora spazio per lavorare.

Implementare una restrizione controllata della dimensionalità aiuta la rete a concentrarsi solo sulle feature discriminanti per le dieci cifre. 
Dobbiamo progettare dei Bottleneck controllati, restringere le feature solo quando la rete ha fatto le feature più inportanti.
La dimensione dell'output di un layer convoluzionale dipende strettamente dalla dimensione del kernel e dallo stride applicato.

Ma la potenza ha un costo in termini di complessità, vediamo come gestirla

Efficienza Parametrica
Massimizzare la capacità
Un'architettura ottimizzata dove bilanciare la profondità con il numero totale di parametri. In MINMST Pro, cerchiaamo di non superare i 500.000 parametri per mantenere il modello agile.
Un modelle più snello generalizza meglio ed impara più infretta.
La potenza di una CNN non sta in quanti pesi ha, ma in come li condivide.
L'impiego di layer 'input' espliciti permette di definire pipeline di preprocessing integrate che normalizzano i dati direttamente nel grafo computazionale.

Regolarizzazione Avanzata
Combattere il rumore nei dati
In un dataset pulito com MNIST, il rischio principale è che il modello impari a memoria le variazioni specifiche dei campioni di training invece di generalizzare il concetto di cifra.
Esploderemo tecniche che stabilizzano l'addestramento e forzano la rete a scoprire feature robuste che non dipendono dalla presenza di singoli neuroni o da specifiche scale di attivazione.
Se il numero 7 che mostriamo alla rete ha sempre un pixel nero in alto a destra, quella volta che la rete vede un pixel bianco (in alto a destra), perchè scritto in modo leggermente diverso, la rete non capire. Dobbiamo quindi essere 'cattivi' con la nostra rete durante la fase ditraining per renderla più forte per la battaglia.
La regolarizzazione avanzata è la nostra palestra.

Stabilizzazione e Generalizzazione
Strumenti di controllo
- Batch Normalization: normalizza gli input di ogni layer per mitigare il problema dell'internal Covariante Shift e accellera la convergenza.
- Spatial Dropout: versione avanzata di dropout che spegne intere feature map di caratteristiche invece dei singoli neuroni, praticolarmente efficace nelle CNNN
- L2 Weight Regularization: penalizzazione dei pesi troppo grandi nella funzione di loss per evitare che il modello diventi eccessivamente sensibile a piccoli cambiamenti nell input
- Data Augmentation: rotazione e traslazioni casuali applicate a runtime per simulare variazioni reali della scrittura manuale.

Logica della Regolarizzazione
Drop Connetc vs Dropout: 
    Mentre il dropout agisce sulle attivazioni, tecniche alternative agiscono sui pesi stessi, creando  una rete più resiliente
Label Smoothing
    Questa tecnica impedisce al modlelo di diventare 'troppo sicuro' delle sue previsioni, ammorbidendo i target One-Hot durante il calcolo della loss. E' come dire alla rete, 'questo numero assomiglia molto a 5' (lasciando un margine di dubbio) invece di dire alla rete 'questo è un 5'. Questo evita che i pesi esplodono aggiungendo una certezza assoluta che nel mondo reale non esiste.
Funzione di Costo Regolarizzata
    La loss finale è la somma della cross-entropy categoriale e del termine di penalità dei pesi controllato dal parametri lambda

Ma attenzione è importante ragionare su come montare questi pezzi insieme. L'ordine è un fattore che cambia il prodotto.

Interazione tra BN e Dropout
Il delicato equilibrio
L'uso simultaneo di Batch Normalization e Dropout richiede attenzione: solitamente è preferibili inserire il dropout dopo la normalizzazione per evitare incoerenze nelle statistiche di batch
La regolarizzazione non deve essere punitiva, l'obbiettivo è spingere la rete verso una 'flag region' della superficie di errore dove la generalizzazione è massima.
L'ordine è:
Convoluzoine -> BN -> attivazione -> dropout

Ora che la nostra macchina è costruita dobbiamo guidarla verso il traguardo.

Raggiungimento del Top Performance
Dettagli che fanno la differenza
L'addestramento di un modello 'Pro' termina solo quando abbiamo spremuto ogni bit di informazione del gradiente. L'ottimizzazione del processo di training è importante quanto l'architettura stessa.
Vedremo come gestire dinamicamente il learning rate e come utilizzare le callback per salvare solo la versione migliore del nostro modello durante le epoche di addestramento.
Dobbiamo rallentare quando siamo vicini alla soluzione (learning rate) per evitare su sorpassare il minimo globale dela loss

Vediamo come sono le strategia per il rellentamento del learning rate controllato

Strategia di Training Loop
Ottimizzazione dinamica
- Learning Rate Decay: riduzione automatica del passo di apprendimento quando la loss di validazione smette di migliorare per rifinire i pesi
- Early Stopping: interruzione del training non appena si rilevano segni di divergenza, risparmiando tempo computazionale e prevedendo l'overfitting
- Model Chackpoint: monitoraggio dell accuratezza su test set per esportare i pesi che hanno ottenuto i miglior punteggio assoluto
- Ensemble Mentale: l'idea che la media di diverse inizzializzazioni posso stabilizzare ulteriormente le previsioni finali.

Ma come leggiamo i dati per capire dove la rete sta ancora sbagliando?

Analisi delle Metriche
- Matrice di Confusione: indispensabile per capire se la rete confonde cifre simili come il 4 e il 9 o il 3 e il 8. In questo caso non serve più potenza ma filtri migliori per le forme verticali.
- Error Analysis Visiva: visualizzare i campioni errati ci permette di capire se il fallimento è dovuto a una scrittura realmente ambigua o a un limite del modello. Si potrebbe scoprire che i casi dove la rete sbaglia sono cifre che perfino un umano farebbe fatica a leggere.
- Decadimento Esponenziale: il tasso di apprendimento al tempo t segue una legge di decadimetno che permette di esplorare inizialmente lo spazio e poi convergere con precisione.

Dove finisce la macchina e dove inizia il limite dei dati?

Verso il 100% di Accuratezza
Il limite umano e del dataset
Raggiungere il 99.5% o superiore significa entrare in una zona dove molti errori sono dovuti a label errate nel dataset originale o a cifre illegibili.
Un modello MNIST è la dimostrazione che con le giuste tecniche di ingegneria, le CNN possono superare la capacità di classificazione umane su task specifici.


In [ ]:
# MNIST è un dataset didattico di 60.000 immagini di training e 10.000 di test, tutte 28x28 in scala di grigi

import os

#
# 1. DEFINIZIONE DEL MOTORE DI CALCOLO (Best Practice 2026)
#
# Nel 2026, l'agnosticismo del framework è la norma. Impostiamo PyTorch come backend.
os.environ["KERAS_BACKEND"] = "torch"

import keras
#import torch # Utilizzato sotto il cofano da Keras, ma non è necessario importarlo

#verifica del backend effettivamente in uso
print("Backend Keras:", keras.config.backend())

#
# 2. CARICAMENTO E PREPARAZIONE "PRO" DEI DATI
#

# Carichiamo MNIST. Teoria: MNIST è il 'Hello World' della computer vision, 
# ma qui lo trattiamo con rigore industriale.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
#ora ogni elemento x_train è una matrice 28x28. Ogni valore rappresenta la luminosità di un pixel (0=nero 255=bianco)
#ogni elemento y_train è invece un numero intero da 0 a 9

# Reshape: Aggiungiamo il canale colore (1 per scala di grigi) richiesto dai layer Conv2D.
# il canale aggiutno vale 1 perchè l'immagine è in scala di grigi, per un'immagine RGB avresti 3 perche i canali sono rosso, verde, blu
# (batch, altezza, larghezza, canali)
# .astype("float32") converte i numeri in decimali a 32 bit, necessario perche il modello lavora con numeri in virgola mobile
# /255.0 normalizza i valori 255->1.0   128->circa 0.502 0->0.0
# questo rende l'ottimizzatore più stabile
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

#
# 3. COSTRUZIONE DEL MODELLO - ARCHITETTURA ESTREMA (BLOCCHI CONV-CONV-POOL)
#

# Teoria: Usare due Conv2D prima del pooling aumenta la non-linearità e il campo ricettivo
# senza ridurre troppo velocemente la risoluzione spaziale (Feature Map preservation).
def build_mnist_pro_model():

    # Inputs
    inputs = keras.Input(shape=(28, 28, 1))
    
    # Blocco 1: Low-level features
    x = keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs) #32=il layer impara 32 filtri; Kernel (3,3) ogni filtro osserva inizialmente una finestra di 3x3 pixel; padding='same' la convoluzione mantiene altezza e larghezza, ingresso 28x28x1 uscita 28x28x1
    x = keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x) # Stabilizza il gradiente durante il backprop
    x = keras.layers.MaxPooling2D((2, 2))(x) #osserva ogni area (2,2) e conserva solo il valore massimo esemio 1,4 e 2,3 mantiene solo 4. La forma passa da 28x28x32 a 14x14x32
    # il pooling riduce il costo computazionale ma elimina anche informazione spaziale. Per questo non conviene applicarlo troppo presto e troppe volte
    x = keras.layers.Dropout(0.2)(x) # Regolarizzazione: previene la co-adattazione dei neuroni

    # Blocco 2: Mid-level features
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2, 2))(x) #parto da 14,14 dopo il pooling ho 7x7, pero' con Conv2D ho aumentato i canali
    #mano mano che la risoluzione diminuisce il modello può rappresentare un numero maggiore di caratteristiche astratte (cos' non perdo informazine)
    x = keras.layers.Dropout(0.3)(x)

    # Blocco 3: High-level features & Global Context
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    
    # Teoria: Il GlobalAveragePooling2D riduce drasticamente i parametri rispetto a un layer Dense,
    # agendo come una potente regolarizzazione strutturale contro l'overfitting.
    x = keras.layers.GlobalAveragePooling2D()(x)
    
    # Classificatore finale
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    
    return keras.Model(inputs, outputs)

model = build_mnist_pro_model()

# 4. COMPILAZIONE E OTTIMIZZATORE
# Utilizziamo AdamW: una variante di Adam che gestisce meglio il weight decay (L2).
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 5. CALLBACKS AVANZATE (Controllo del Top Performance)
# Strategia: Se la loss non scende, riduciamo la velocità (LR) per 'rifinire' i pesi.
callbacks = [
    # Riduce il LR se la val_loss smette di migliorare (fine-tuning dinamico)
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    # Ferma il training prima che inizi l'overfitting
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    # Salvataggio atomico nel formato standard 2026 (.keras)
    keras.callbacks.ModelCheckpoint("mnist_pro_model.keras", save_best_only=True)
]

# 6. TRAINING LOOP
print("[INFO] Addestramento MNIST Pro in corso...")
history = model.fit(
    x_train, y_train,
    epochs=50, # EarlyStopping gestirà la durata reale
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# 7. VALUTAZIONE FINALE
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n[RISULTATO] Accuratezza sul Test Set: {acc*100:.2f}%")

Backend Keras: torch
[INFO] Addestramento MNIST Pro in corso...
Epoch 1/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.9323 - loss: 0.2245 - val_accuracy: 0.4258 - val_loss: 2.3972 - learning_rate: 0.0010
Epoch 2/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.9821 - loss: 0.0591 - val_accuracy: 0.9892 - val_loss: 0.0363 - learning_rate: 0.0010
Epoch 3/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 40ms/step - accuracy: 0.9866 - loss: 0.0455 - val_accuracy: 0.9812 - val_loss: 0.0711 - learning_rate: 0.0010
Epoch 4/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.9897 - loss: 0.0360 - val_accuracy: 0.9903 - val_loss: 0.0332 - learning_rate: 0.0010
Epoch 5/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.9908 - loss: 0.0314 - val_accuracy: 0.9935 - val_loss: 0.0235 - learning_rate: 0.0010
Epoch 6/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 41ms/step - accuracy: 0.9911 - loss: 0.0290 - val_accuracy: 0.9928 - val_loss: 0.0253 - learning_rate: 0.0010
Epoch 7/50
4